# ENGE707 — Geometry-adversarial model: Kaggle test submission

**Inference only — no training.** Loads `best_geometry_adversarial.pt` saved during the 99.60% validation run, uses exactly the validation image processing from the trained model (deterministic reflective square framing, bicubic resize, RGB ImageNet normalisation), and predicts the test image IDs listed in `sample_submission.csv`.

Outputs two valid alternative submissions to compare:
- `submission_geometry_adversarial.csv` — one original view
- `submission_geometry_adversarial_flip_tta.csv` — average probabilities from original and horizontally flipped view

**Do not use `sample_submission.csv` labels as predictions**; the values there are placeholders. Test images are never read for fitting, statistics, or validation.

Run this notebook directly from `C:\AI Workspace\Kaggle` (your existing VS Code workspace), preferably in a restarted kernel to recover memory after training. Expected test rows with the provided file: 29,443.


In [2]:
# 1 — Paths and imports: run from your Kaggle project root
from pathlib import Path
from collections import Counter
import re
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms as T
from torchvision.transforms import InterpolationMode

ROOT = Path.cwd()
TEST_DIR = ROOT / "test"                # nested test/test is found automatically
SAMPLE_CSV = ROOT / "sample_submission.csv"
CHECKPOINT = ROOT / "outputs" / "robust_recognition_pipeline" / "best_geometry_adversarial.pt"
OUT = ROOT / "outputs" / "robust_recognition_pipeline"
OUT.mkdir(parents=True, exist_ok=True)
BATCH_SIZE = 16                          # reduce to 8 if GPU runs out of memory
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

assert SAMPLE_CSV.is_file(), f"Missing {SAMPLE_CSV}"
assert CHECKPOINT.is_file(), f"Missing saved checkpoint: {CHECKPOINT}"
assert TEST_DIR.is_dir(), f"Missing test directory: {TEST_DIR}"
print("Device:", DEVICE, "| checkpoint:", CHECKPOINT)
print("Test root:", TEST_DIR)


Device: cuda | checkpoint: c:\AI Workspace\Kaggle\outputs\robust_recognition_pipeline\best_geometry_adversarial.pt
Test root: c:\AI Workspace\Kaggle\test


In [3]:
# 2 — Rebuild the exact network and load the BEST trained weights
# We do not download pretrained weights: the checkpoint includes the backbone and all heads.
class GradientReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, coefficient):
        ctx.coefficient = float(coefficient)
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.coefficient * grad_output, None

def create_backbone(name):
    options = {
        "convnext_tiny": models.convnext_tiny,
        "resnet50": models.resnet50,
        "efficientnet_v2_s": models.efficientnet_v2_s,
    }
    if name not in options:
        raise ValueError(f"Unsupported checkpoint backbone: {name}")
    backbone = options[name](weights=None)
    if name == "resnet50":
        dim = backbone.fc.in_features
        backbone.fc = nn.Identity()
    else:
        dim = backbone.classifier[-1].in_features
        backbone.classifier[-1] = nn.Identity()
    return backbone, dim

class DebiasedFacialNet(nn.Module):
    def __init__(self, name, n_size, n_aspect, n_classes):
        super().__init__()
        self.backbone, dim = create_backbone(name)
        self.classifier = nn.Sequential(nn.Dropout(.15), nn.Linear(dim, n_classes))
        self.size_head = nn.Sequential(nn.Linear(dim,128), nn.ReLU(), nn.Linear(128,n_size))
        self.aspect_head = nn.Sequential(nn.Linear(dim,128), nn.ReLU(), nn.Linear(128,n_aspect))
    def forward(self, x):
        return self.classifier(self.backbone(x))  # geometry heads unused for inference

# Only load checkpoint files you created/trust (torch.load unpickles their contents).
ckpt = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
CLASSES = list(ckpt["classes"])
MODEL_NAME = ckpt["model_name"]
IMAGE_SIZE = int(ckpt["image_size"])
MEAN, STD = tuple(ckpt["mean"]), tuple(ckpt["std"])
assert len(CLASSES) == 8 and len(set(CLASSES)) == 8

net = DebiasedFacialNet(MODEL_NAME, len(ckpt["size_edges"])+1,
                        len(ckpt["aspect_edges"])+1, len(CLASSES))
net.load_state_dict(ckpt["model"], strict=True)
net = net.to(DEVICE).eval()
print("Loaded epoch:", ckpt["epoch"]+1, "| best validation:", ckpt["best_accuracy"])
print("Backbone:", MODEL_NAME, "| size:", IMAGE_SIZE, "| class order:", CLASSES)


Loaded epoch: 5 | best validation: 0.9959943449575872
Backbone: convnext_tiny | size: 224 | class order: ['CatEye', 'CatFace', 'CatMouth', 'CatNose', 'DogEye', 'DogFace', 'DogMouth', 'DogNose']


In [4]:
# 3 — Deterministic image preprocessing, matching training validation EXACTLY
# Preserve aspect ratio via reflected square padding; no random crops or colour transforms.
class ReflectSquare:
    def __call__(self, image):
        a = np.asarray(image.convert("RGB"))
        h, w = a.shape[:2]
        side = max(h, w)
        dx, dy = side-w, side-h
        pad = ((dy//2,dy-dy//2), (dx//2,dx-dx//2), (0,0))
        return Image.fromarray(np.pad(a, pad, mode="reflect" if min(h,w)>1 else "edge"))

transform = T.Compose([
    ReflectSquare(),
    T.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=InterpolationMode.BICUBIC, antialias=True),
    T.ToTensor(), T.Normalize(mean=MEAN, std=STD)
])

sample = pd.read_csv(SAMPLE_CSV)
assert list(sample.columns) == ["image_id", "class"], f"Unexpected sample schema: {sample.columns.tolist()}"
assert not sample.image_id.duplicated().any(), "Duplicate IDs in sample submission"
sample["image_id"] = pd.to_numeric(sample.image_id, errors="raise").astype(int)

# Scan ONLY the Kaggle test directory; training/test IDs can overlap.
valid_extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}
test_index = {}
for p in TEST_DIR.rglob("*"):
    if not p.is_file() or p.suffix.lower() not in valid_extensions:
        continue
    match = re.search(r"(\d+)$", p.stem)
    if match:
        image_id = int(match.group(1))
        if image_id in test_index:
            raise ValueError(f"Multiple test images for ID {image_id}: {test_index[image_id]} and {p}")
        test_index[image_id] = p

missing = sorted(set(sample.image_id) - set(test_index))
assert not missing, f"Missing {len(missing)} test images; first 10 IDs: {missing[:10]}"
print(f"Matched {len(sample):,} submission rows with test images. Found {len(test_index):,} indexed files.")

class KaggleTestDataset(Dataset):
    def __len__(self): return len(sample)
    def __getitem__(self, i):
        image_id = int(sample.iloc[i]["image_id"])
        with Image.open(test_index[image_id]) as im:
            return transform(im.convert("RGB"))

test_loader = DataLoader(KaggleTestDataset(), batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=0, pin_memory=(DEVICE.type=="cuda"))


Matched 29,443 submission rows with test images. Found 29,443 indexed files.


In [5]:
# 4 — Predict original view AND flip-TTA; write two submission CSVs
# Both files preserve EXACTLY the sample_submission.csv row ordering.
base_preds, tta_preds = [], []
with torch.inference_mode():
    for step, x in enumerate(test_loader, start=1):
        x = x.to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type=="cuda")):
            p_original = torch.softmax(net(x).float(), dim=1)
            p_flipped = torch.softmax(net(torch.flip(x, dims=[3])).float(), dim=1)
        base_preds.extend(p_original.argmax(dim=1).cpu().tolist())
        tta_preds.extend(((p_original+p_flipped)/2).argmax(dim=1).cpu().tolist())
        if step % 100 == 0 or step == len(test_loader):
            print(f"Processed {min(step*BATCH_SIZE,len(sample)):,}/{len(sample):,} test images")

assert len(base_preds) == len(tta_preds) == len(sample)
plain = sample[["image_id"]].copy()
plain["class"] = [CLASSES[i] for i in base_preds]
tta = sample[["image_id"]].copy()
tta["class"] = [CLASSES[i] for i in tta_preds]
assert set(plain["class"]).issubset(CLASSES)
assert set(tta["class"]).issubset(CLASSES)

plain_file = OUT / "submission_geometry_adversarial.csv"
tta_file = OUT / "submission_geometry_adversarial_flip_tta.csv"
plain.to_csv(plain_file, index=False)
tta.to_csv(tta_file, index=False)
print("\nSaved original-view submission:", plain_file)
print("Saved flip-TTA submission:", tta_file)
print("Prediction changes under flip-TTA:", sum(a!=b for a,b in zip(base_preds,tta_preds)))
print("\nClass counts (flip-TTA):\n", tta["class"].value_counts().to_string())
print("\nSubmission preview:\n", tta.head().to_string(index=False))


Processed 1,600/29,443 test images
Processed 3,200/29,443 test images
Processed 4,800/29,443 test images
Processed 6,400/29,443 test images
Processed 8,000/29,443 test images
Processed 9,600/29,443 test images
Processed 11,200/29,443 test images
Processed 12,800/29,443 test images
Processed 14,400/29,443 test images
Processed 16,000/29,443 test images
Processed 17,600/29,443 test images
Processed 19,200/29,443 test images
Processed 20,800/29,443 test images
Processed 22,400/29,443 test images
Processed 24,000/29,443 test images
Processed 25,600/29,443 test images
Processed 27,200/29,443 test images
Processed 28,800/29,443 test images
Processed 29,443/29,443 test images

Saved original-view submission: c:\AI Workspace\Kaggle\outputs\robust_recognition_pipeline\submission_geometry_adversarial.csv
Saved flip-TTA submission: c:\AI Workspace\Kaggle\outputs\robust_recognition_pipeline\submission_geometry_adversarial_flip_tta.csv
Prediction changes under flip-TTA: 153

Class counts (flip-TTA)